# Soldani - Complete report pipeline (FairMind and LLM, side by side)

This notebook extends `2_6_report_pipeline.ipynb` with the workflow requested by the
supervisor on 20 August 2026. The same Bayesian network feeds two branches:

- **Branch A**: FairMind computes the five effects, the model writes only the
  interpretation. This is what `2_6` already did.
- **Branch B**: the model receives the pre-aggregated probability tables and computes
  the five effects itself, as in `2_3_benchmark_thor.ipynb`. Its numbers are then fed
  into the same report template, and the resulting report is scored as well.

The question the second branch answers is whether a numerical error propagates into the
interpretation. A model that is wrong on DE by a third may still answer all five recap
questions correctly, if the error does not cross a threshold. Measuring that is the point.

Shared functions come from `src/benchmark_common.py`, so that both branches use exactly
the same reference computation and the same prompt builders as the earlier notebooks.

## 1. Setup

In [1]:
from pathlib import Path
import sys

# Find the root by searching the "src" folder
current = Path.cwd()

while current != current.parent:
    if (current / "src").exists():
        REPO_ROOT = current
        break
    current = current.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

## 2. Imports

`run_fairmind`, `build_llm_prompt` and `compute_discrepancies` are imported from
`src/benchmark_common.py` rather than redefined here. They used to be duplicated across
the benchmark notebooks, and the two misalignments described in the experimental chapter
originated in exactly that duplication: corrected in one notebook, still present in the
others.

In [2]:
import datetime
import json
import os

import pandas as pd

from src.llm import LLM_CONFIGS, call_llm
from src.benchmark_common import build_llm_prompt, compute_discrepancies, run_fairmind

from src.report_pipeline.prompt_builder import build_prompts
from src.report_pipeline.llm_client import call_llm_report, find_unfilled_placeholders
from src.report_pipeline.validator import (
    score_report, GROUND_TRUTH_RULES,
    THRESH_DE, EPS_IE, THRESH_TV, THRESH_SE_REL,
)
from src.report_pipeline.annotate import annotate_recap_answers

LLAMA_HOST = os.environ.get("LLAMA_HOST", "localhost")
LLAMA_PORT = os.environ.get("LLAMA_PORT", "8080")
LLM_CONFIGS[0]["base_url"] = f"http://{LLAMA_HOST}:{LLAMA_PORT}/v1"

print(f"LLM endpoint configured: http://{LLAMA_HOST}:{LLAMA_PORT}/v1")

LLM endpoint configured: http://gnode04:8080/v1


## 3. Configuration

The same `CONFIG` as `2_3` and `2_6`: Adult, protected attribute `S2_gender`
(Female to Male), target `T_income` (`>50K`), mediator `hours-per-week`, confounder
`education`.

In [3]:
CONFIG = {
    "dataset_name": "adult",
    "csv_path": "../../data/processed/adult.csv",
    "target_col":  "T_income",
    "target_val":  ">50K",
    "protected":   "S2_gender",
    "x0": "Female",
    "x1": "Male",
    "mediators":   ["hours-per-week"],
    "confounders": ["education"],
    # The discretisation is declared here rather than written into
    # run_fairmind, so that the same function serves every dataset. "cut"
    # style specs give numeric bands, "mapping" collapses categorical levels.
    # Sixteen education levels by five hour bands would be eighty (z,w) pairs,
    # which the model could not enumerate without truncating; five tiers bring
    # them down to twenty five.
    "binning": {
        "hours-per-week": {
            "bins": [0, 20, 35, 45, 60, 100],
            "labels": ["<=20", "21-35", "36-45", "46-60", ">60"],
        },
        "education": {
            "mapping": {
                "Preschool": "<HS", "1st-4th": "<HS", "5th-6th": "<HS",
                "7th-8th": "<HS", "9th": "<HS", "10th": "<HS",
                "11th": "<HS", "12th": "<HS",
                "HS-grad": "HS-grad",
                "Some-college": "Some-college", "Assoc-acdm": "Some-college",
                "Assoc-voc": "Some-college",
                "Bachelors": "Bachelors",
                "Masters": "Grad", "Prof-school": "Grad", "Doctorate": "Grad",
            }
        },
    },
}

DRY_RUN = False   # True = no network call, simulated output (NOT a valid result)

## 4. Common step: FairMind computes the reference values

Both branches start from this network. Branch A uses the effects, branch B queries the
same fitted network to build its probability tables, so the two sides start from
identical numbers on every cell.

In [4]:
ground_truth, bn, n_rows, fairmind_time = run_fairmind(CONFIG)

print(f"FairMind - elapsed time: {fairmind_time:.4f}s  ({n_rows} rows)")
for k, v in ground_truth.items():
    print(f"  {k}: {v:.6f}")

# Eq. 9 closes on the reverse form only; a mismatch here means the two IE
# definitions have been mixed up again.
check = ground_truth["DE"] - ground_truth["IE_reverse"]
print(f"\n  decomposition check: DE - IE_reverse = {check:.6f}  vs  TE = {ground_truth['TE']:.6f}")

2026-08-22 00:56:07.643 | DEBUG    | src.model:fit_discrete_bayesian_model:33 - Using estimator: <class 'pgmpy.estimators.BayesianEstimator.BayesianEstimator'> with parameters: {'prior_type': 'dirichlet', 'pseudo_counts': 1}


INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'S2_gender': 'C', 'hours-per-week': 'O', 'education': 'C', 'T_income': 'C'}


2026-08-22 00:56:07.701 | DEBUG    | src.effects:total_variation:248 - Computing total variation for target=('T_income', '>50K'), private_baseline=Female, private_mod=Male


FairMind - elapsed time: 0.0190s  (48842 rows)


  TV: 0.193714
  TE: 0.184736
  SE: 0.008977
  DE: 0.138404
  IE: 0.016869
  IE_reverse: -0.046333

  decomposition check: DE - IE_reverse = 0.184736  vs  TE = 0.184736


## 5. Branch A: the model interprets numbers it did not compute

The five exact values are injected into the LaTeX template in Python, before the model
sees the document. Only the eight qualitative placeholders are left to fill.

The row for the indirect effect carries the additive form, `-IE_reverse`, following the
convention confirmed by the supervisor: with `TE = DE + IE`, the sign of IE relative to DE
says directly whether the mediator amplifies or offsets the disparity.

In [5]:
REPORT_DATE = datetime.date.today().isoformat()

context = {
    "dataset": CONFIG["dataset_name"],
    "protected_attr": CONFIG["protected"],
    "x0": CONFIG["x0"],
    "x1": CONFIG["x1"],
    "outcome_attr": f"{CONFIG['target_col']} ({CONFIG['target_val']})",
    "mediator": ", ".join(CONFIG["mediators"]),
    "confounder": ", ".join(CONFIG["confounders"]),
}

effects_A = {**ground_truth, "IE": -ground_truth["IE_reverse"]}
system_prompt_A, user_prompt_A = build_prompts(effects_A, context, REPORT_DATE)

print("Placeholder left to LLM:", find_unfilled_placeholders(user_prompt_A))

Placeholder left to LLM: ['QUALITATIVE_TOTAL', 'QUALITATIVE_DE', 'QUALITATIVE_IE', 'ANSWER_Q1', 'ANSWER_Q2', 'ANSWER_Q3', 'ANSWER_Q4', 'ANSWER_Q5']


In [6]:
def simulate_report(user_prompt):
    """Fill the placeholders locally, for the dry run only."""
    from src.report_pipeline.llm_client import extract_latex_document
    mock = extract_latex_document(user_prompt)
    for ph in ["QUALITATIVE_TOTAL", "QUALITATIVE_DE", "QUALITATIVE_IE"]:
        mock = mock.replace(f"<<{ph}>>", "[SIMULATED TEXT - DRY RUN, not real content]")
    for i in range(1, 6):
        mock = mock.replace(f"<<ANSWER_Q{i}>>", "YES")
    return mock


if DRY_RUN:
    report_A = simulate_report(user_prompt_A)
    usage_A = {"input_tokens": None, "output_tokens": None,
               "total_tokens": None, "finish_reason": "dry_run"}
    time_A = 0.0
    print("DRY RUN - no call to the LLM was made.")
else:
    report_A, usage_A, time_A = call_llm_report(
        system_prompt_A, user_prompt_A, max_tokens=4096, cache_prompt=False,
    )
    print(f"Branch A - time: {time_A:.2f}s, "
          f"tokens {usage_A['input_tokens']}/{usage_A['output_tokens']} "
          f"(finish_reason={usage_A['finish_reason']})")

leftover_A = find_unfilled_placeholders(report_A)
print("Unfilled placeholders:", leftover_A if leftover_A else "none")

INFO:httpx:HTTP Request: POST http://gnode04:8080/v1/chat/completions "HTTP/1.1 200 OK"


Branch A - time: 12.06s, tokens 1422/878 (finish_reason=stop)
Unfilled placeholders: none


## 6. Branch B: the model computes the effects itself

The prompt is the one from `2_3`: the model receives the probability tables queried from
the same fitted network, and is asked to apply the identification formulas. The spurious
effect is not requested, since it is fully determined by TV and TE; it is derived here
with the same identity used for the reference values, so its comparison reflects only the
model's errors on TV and TE.

In [7]:
prompt_B = build_llm_prompt(CONFIG, bn, n_rows)
print(f"Computation prompt: {len(prompt_B)} characters")

if DRY_RUN:
    # Deliberately imperfect, so that the discrepancy table is not trivially zero.
    llm_effects = {"TV": 0.1937, "TE": 0.1849, "DE": 0.1840, "IE": 0.0180}
    usage_B = {"input_tokens": None, "output_tokens": None, "total_tokens": None}
    time_B = 0.0
    print("DRY RUN - simulated values, NOT a valid result.")
else:
    llm_effects, usage_B, time_B = call_llm(prompt_B, max_tokens=16384, cache_prompt=False)
    print(f"Branch B - time: {time_B:.2f}s, "
          f"tokens {usage_B['input_tokens']}/{usage_B['output_tokens']}")

# Same identity as the reference: SE = TV - TE.
llm_effects["SE"] = llm_effects["TV"] - llm_effects["TE"]
llm_effects = {k: float(llm_effects[k]) for k in ["TV", "TE", "SE", "DE", "IE"]}
print(json.dumps(llm_effects, indent=2))

Computation prompt: 6201 characters


INFO:httpx:HTTP Request: POST http://gnode04:8080/v1/chat/completions "HTTP/1.1 200 OK"


Branch B - time: 131.07s, tokens 2928/9701
{
  "TV": 0.1937,
  "TE": 0.184852,
  "SE": 0.008848000000000023,
  "DE": 0.184,
  "IE": 0.018
}


### Discrepancies against the reference values

Only the five effects the model is asked for. `IE_reverse` stays out: comparing an answer
against an estimand nobody requested measures nothing.

In [8]:
discrepancies = compute_discrepancies(ground_truth, llm_effects)
print(discrepancies.to_string(index=False))

effect  fairmind      llm  abs_error  rel_error_%
    TV  0.193714 0.193700   0.000014         0.01
    TE  0.184736 0.184852   0.000116         0.06
    SE  0.008977 0.008848   0.000129         1.44
    DE  0.138404 0.184000   0.045596        32.94
    IE  0.016869 0.018000   0.001131         6.71


## 7. Branch B, second stage: a report built on the model's own numbers

The same template is filled again, this time with the values the model produced. One
quantity needs care. The model computes IE in the direct form of Eq. 8, which is not the
form the report displays, and not the one the recap rules read. Both are derived from the
model's own TE and DE, using the additive convention `TE = DE + IE`:

- displayed in the report: `IE = TE - DE`
- read by the scoring rules: `IE_reverse = DE - TE`

Deriving them, rather than asking the model for a third quantity, keeps the report
faithful to the scenario it represents: a document built entirely on the model's numbers,
with no reference value leaking in.

In [9]:
# Both forms derived from the model's own TE and DE.
ie_additive_llm = llm_effects["TE"] - llm_effects["DE"]
ie_reverse_llm = llm_effects["DE"] - llm_effects["TE"]

# Sanity check on the reference values: the same derivation must reproduce
# FairMind's IE_reverse exactly.
_check = ground_truth["DE"] - ground_truth["TE"]
print(f"derivation check on the reference values: {_check:.9f} "
      f"vs IE_reverse {ground_truth['IE_reverse']:.9f}")

effects_B = {**llm_effects, "IE": ie_additive_llm}
system_prompt_B, user_prompt_B = build_prompts(effects_B, context, REPORT_DATE)

if DRY_RUN:
    report_B = simulate_report(user_prompt_B)
    usage_B2 = {"input_tokens": None, "output_tokens": None,
                "total_tokens": None, "finish_reason": "dry_run"}
    time_B2 = 0.0
else:
    report_B, usage_B2, time_B2 = call_llm_report(
        system_prompt_B, user_prompt_B, max_tokens=4096, cache_prompt=False,
    )
    print(f"Branch B report - time: {time_B2:.2f}s, "
          f"tokens {usage_B2['input_tokens']}/{usage_B2['output_tokens']}")

leftover_B = find_unfilled_placeholders(report_B)
print("Unfilled placeholders:", leftover_B if leftover_B else "none")

derivation check on the reference values: -0.046332531 vs IE_reverse -0.046332531


INFO:httpx:HTTP Request: POST http://gnode04:8080/v1/chat/completions "HTTP/1.1 200 OK"


Branch B report - time: 11.33s, tokens 1422/856
Unfilled placeholders: none


## 8. Scoring

Three scores, answering three different questions.

| report | scored against | question |
|---|---|---|
| A | FairMind | does the model interpret correct numbers correctly? |
| B | FairMind | does a numerical error propagate into the interpretation? |
| B | the model's own numbers | is the report consistent with the values it was given? |

The middle row is the one the supervisor asked for. A report can be internally consistent
and still answer wrongly in absolute terms, if the numbers it was built on are wrong.

In [10]:
print("Thresholds used for the ground truth:")
print(f"  THRESH_DE     = {THRESH_DE}    (|DE| above => direct discrimination)")
print(f"  EPS_IE        = {EPS_IE}   (|IE| below => negligible channel)")
print(f"  THRESH_TV     = {THRESH_TV}    (|TV| above => practical relevance)")
print(f"  THRESH_SE_REL = {THRESH_SE_REL}    (|SE|/|TV| above => substantial spurious part)")
print()

# Reference dictionary for the rules: needs IE_reverse.
llm_effects_for_rules = {**llm_effects, "IE_reverse": ie_reverse_llm}

score_A_vs_fairmind = score_report(report_A, ground_truth)
score_B_vs_fairmind = score_report(report_B, ground_truth)
score_B_vs_llm      = score_report(report_B, llm_effects_for_rules)

for label, s in [
    ("A scored against FairMind", score_A_vs_fairmind),
    ("B scored against FairMind", score_B_vs_fairmind),
    ("B scored against its own numbers", score_B_vs_llm),
]:
    r = s.to_dict()
    print(f"  {label:36} {r['n_correct']}/{r['n_total']}  = {r['score_pct']}")

Thresholds used for the ground truth:
  THRESH_DE     = 0.05    (|DE| above => direct discrimination)
  EPS_IE        = 0.005   (|IE| below => negligible channel)
  THRESH_TV     = 0.05    (|TV| above => practical relevance)
  THRESH_SE_REL = 0.1    (|SE|/|TV| above => substantial spurious part)

  A scored against FairMind            5/5  = 100.0%
  B scored against FairMind            5/5  = 100.0%
  B scored against its own numbers     5/5  = 100.0%


### Question by question

In [11]:
comparison = pd.DataFrame({
    "question": [q["index"] for q in score_A_vs_fairmind.to_dict()["questions"]],
    "A_answer": [q["llm_answer"] for q in score_A_vs_fairmind.to_dict()["questions"]],
    "A_expected": [q["ground_truth"] for q in score_A_vs_fairmind.to_dict()["questions"]],
    "B_answer": [q["llm_answer"] for q in score_B_vs_fairmind.to_dict()["questions"]],
    "B_expected_fairmind": [q["ground_truth"] for q in score_B_vs_fairmind.to_dict()["questions"]],
    "B_expected_own": [q["ground_truth"] for q in score_B_vs_llm.to_dict()["questions"]],
})
print(comparison.to_string(index=False))

# Where the two reference dictionaries disagree, the model's numerical error has
# crossed a threshold and changed the correct answer.
flipped = comparison[comparison["B_expected_fairmind"] != comparison["B_expected_own"]]
print()
if flipped.empty:
    print("No question flips: the numerical error stays within the thresholds.")
else:
    print(f"Questions whose expected answer flips: {list(flipped['question'])}")

 question A_answer A_expected B_answer B_expected_fairmind B_expected_own
        1      YES        YES      YES                 YES            YES
        2       NO         NO       NO                  NO             NO
        3      YES        YES      YES                 YES            YES
        4       NO         NO       NO                  NO             NO
        5      YES        YES      YES                 YES            YES

No question flips: the numerical error stays within the thresholds.


## 9. Saving the artefacts

Both reports are written to disk, together with their annotated versions and a single
JSON holding the reference values, the model's values, the discrepancies and the three
scores. The `dry_run` flag distinguishes an offline rehearsal from a real execution
beyond any doubt.

In [12]:
os.makedirs("benchmark_results/complete", exist_ok=True)
ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
stem = f"benchmark_results/complete/{CONFIG['dataset_name']}_{ts}"

paths = {}
for tag, report, score in [
    ("A_fairmind_numbers", report_A, score_A_vs_fairmind),
    ("B_llm_numbers", report_B, score_B_vs_fairmind),
]:
    tex_path = f"{stem}_{tag}.tex"
    with open(tex_path, "w", encoding="utf-8") as f:
        f.write(report)
    annotated_path = f"{stem}_{tag}_annotated.tex"
    with open(annotated_path, "w", encoding="utf-8") as f:
        f.write(annotate_recap_answers(report, score))
    paths[tag] = tex_path
    print(f"saved: {tex_path}")

saved: benchmark_results/complete/adult_20260822_005842_A_fairmind_numbers.tex
saved: benchmark_results/complete/adult_20260822_005842_B_llm_numbers.tex


In [13]:
out = {
    "dataset": CONFIG["dataset_name"],
    "timestamp": ts,
    "dry_run": DRY_RUN,
    "config": {k: v for k, v in CONFIG.items() if k != "csv_path"},
    "n_rows": n_rows,
    "fairmind_effects": ground_truth,
    "llm_effects": llm_effects,
    "llm_derived": {
        "IE_additive_shown_in_report": ie_additive_llm,
        "IE_reverse_used_by_rules": ie_reverse_llm,
    },
    "ie_form_used_for_scoring": "IE_reverse",
    "decomposition_check": {
        "DE_minus_IE_reverse": round(ground_truth["DE"] - ground_truth["IE_reverse"], 6),
        "TE": round(ground_truth["TE"], 6),
    },
    "discrepancies": discrepancies.to_dict(orient="records"),
    "scoring": {
        "A_vs_fairmind": score_A_vs_fairmind.to_dict(),
        "B_vs_fairmind": score_B_vs_fairmind.to_dict(),
        "B_vs_llm_own": score_B_vs_llm.to_dict(),
    },
    "unfilled_placeholders": {"A": leftover_A, "B": leftover_B},
    "thresholds": {
        "THRESH_DE": THRESH_DE, "EPS_IE": EPS_IE,
        "THRESH_TV": THRESH_TV, "THRESH_SE_REL": THRESH_SE_REL,
    },
    "token_usage": {
        "A_report": usage_A,
        "B_computation": usage_B,
        "B_report": usage_B2,
    },
    "timing": {
        "fairmind_seconds": round(fairmind_time, 4),
        "A_report_seconds": round(time_A, 4),
        "B_computation_seconds": round(time_B, 4),
        "B_report_seconds": round(time_B2, 4),
    },
    "reports": paths,
}

json_path = f"{stem}.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(out, f, indent=2, ensure_ascii=False, default=float)
print(f"Results saved: {json_path}")

Results saved: benchmark_results/complete/adult_20260822_005842.json


## 10. Summary

The two branches differ in cost by roughly an order of magnitude, and the comparison
between the three scores is what the chapter reports. Two readings are possible and only
the data decides between them: either the numerical error stays inside the thresholds,
and the interpretation survives it, or it crosses one, and a wrong number becomes a wrong
answer.

In [14]:
summary = pd.DataFrame([
    {"branch": "A: FairMind computes",
     "seconds": round(time_A, 2),
     "output_tokens": usage_A.get("output_tokens"),
     "score_vs_fairmind": score_A_vs_fairmind.to_dict()["score_pct"],
     "score_vs_own": "n/a"},
    {"branch": "B: model computes",
     "seconds": round(time_B + time_B2, 2),
     "output_tokens": (usage_B.get("output_tokens") or 0) + (usage_B2.get("output_tokens") or 0)
                      if usage_B.get("output_tokens") is not None else None,
     "score_vs_fairmind": score_B_vs_fairmind.to_dict()["score_pct"],
     "score_vs_own": score_B_vs_llm.to_dict()["score_pct"]},
])
print(summary.to_string(index=False))

              branch  seconds  output_tokens score_vs_fairmind score_vs_own
A: FairMind computes    12.06            878            100.0%          n/a
   B: model computes   142.40          10557            100.0%       100.0%
